In [1]:

import asyncio
import warnings
import pandas as pd
from pathlib import Path

# Парсеры
from antimony import antimony_parser
from westmetall import westmetall_async
from lme import lme_selenium_async
from lbma import lbma_prescious_async
from cbr import cb_currency, cb_metalls
from nbk import nbk_tenge_async
from shmet import shmet_optimized_async
from kitco import kitco_parser_async

# Сервисные функции из service_layer
from service_layer import (
    read_db,
    save_db,
    check_df,
    check_and_save_pair,
    show_db,
    excel_to_csv_db
)

warnings.filterwarnings("ignore")


# ================================================================
# Пути к базам (реальная структура проекта)
# ================================================================
LME_PATH = Path("lme/data/LME_db_new.xlsx")
WESTMETALL_PATH = Path("westmetall/data/LME_westmetall_db.xlsx")

KITCO_PATH = Path("kitco/data/kitko_db.xlsx")
LBMA_PATH = Path("lbma/data/lbma_kitco_subs.xlsx")

ANTIMONY_PATH = Path("antimony/data/antimony.xlsx")

CB_CURRENCY_PATH = Path("cbr/data/cb_currency.xlsx")
CB_METALLS_PATH = Path("cbr/data/cb_metalls.xlsx")

NBK_PATH = Path("nbk/data/nbk_tenge.xlsx")
SHMET_PATH = Path("shmet/data/shmet_historical.xlsx")


# ================================================================
# Проверка целостности парных баз
# ================================================================
def db_check():
    """
    Проверка целостности парных баз с очисткой дубликатов
    и сохранением первого (более раннего) вхождения.
    """
    print("Проверка LME / Westmetall...")
    check_and_save_pair(
        LME_PATH,
        WESTMETALL_PATH,
        pair_name="LME / Westmetall",
        index=False,
    )

    print("Проверка Kitco / LBMA...")
    check_and_save_pair(
        KITCO_PATH,
        LBMA_PATH,
        pair_name="Kitco / LBMA",
        index=False,
    )
    
async def main():
    print("Parsing started...")

    tasks = {
        "lme": lme_selenium_async(),
        "antimony": antimony_parser(),
        "westmetall": westmetall_async(),
        "lbma": lbma_prescious_async(),
        "kitco": kitco_parser_async(),
        "cb_currency": cb_currency(),
        "cb_metalls": cb_metalls(),
        "nbk": nbk_tenge_async(),
        "shmet": shmet_optimized_async(),
    }

    # return_exceptions=True — чтобы падение одного парсера
    # не останавливало остальные
    results = await asyncio.gather(
        *tasks.values(),
        return_exceptions=True,
    )

    for name, result in zip(tasks.keys(), results):
        if isinstance(result, Exception):
            print(f"❌ Ошибка в {name}: {result}")

    print("All tasks are done!")
    print("+" * 64)
    print("Checking DB...")

    db_check()

    print("DB check completed!")
    print("+" * 64)
    print("Visual control")
    print("+" * 64)
    
    print("Converting Excel to CSV...")
    excel_to_csv_db()
    print("CSV conversion completed!")

    # Базовые металлы
    show_db("lme_selenium_db", LME_PATH, sheet_name=0)
    show_db("westmetall_db", WESTMETALL_PATH, sheet_name=0)

    # Драгоценные металлы
    show_db("kitco_db", KITCO_PATH, sheet_name=0)
    show_db("lbma_precious_db", LBMA_PATH, sheet_name=0)

    # Антимоний
    show_db("antimony_db", ANTIMONY_PATH, sheet_name=0)

    # ЦБ РФ: валюты (каждая на своем листе)
    for currency in [
        "USD",
        "EUR",
        "British_Pound",
        "China_Yuan",
        "Japanese_Yen",
        "Swiss_Franc",
    ]:
        show_db(
            f"cb_currency ({currency})",
            CB_CURRENCY_PATH,
            sheet_name=currency,
        )

    # ЦБ РФ: металлы
    show_db("cb_metalls_db", CB_METALLS_PATH, sheet_name=0)

    # Казахстан и SHMET
    show_db("nbk_tenge_db", NBK_PATH, sheet_name=0)
    show_db("shmet_historical_db", SHMET_PATH, sheet_name=0, show_head=True)


# Для Jupyter используем await, а не asyncio.run()
await main()

Parsing started...
🚀 LME parsing started...
antimony parsing is DONE
NBK_tenge parsing is DONE! (1689 строк)
CB_metalls parsing is DONE!
USD is done!
EUR is done!
Australian_Dollar is done!
China_Yuan is done!
British_Pound is done!
Kazakhstan_Tenge is done!
Japanese_Yen is done!
Swiss_Franc is done!
CB_currency parsing is DONE!
✅ LME_main is done!!!
WESTMETALL is done!!!
Произошла ошибка KITCO: Не найдены блоки <div class='grid'> на странице Kitco
LBMA is done!!!
SHMET is done!!!
All tasks are done!
++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
Checking DB...
Проверка LME / Westmetall...
LME / Westmetall: добавлены пропущенные даты и удалены дубликаты
Проверка Kitco / LBMA...
Kitco / LBMA: добавлены пропущенные даты и удалены дубликаты
DB check completed!
++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
Visual control
++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
Converting Excel to CSV...
Поиск Excel файлов...
Найдено 9 Excel файл

,date,aluminium,copper,lead,nickel,zink,tin
1187,2026-09-10,3343.0,14390.0,1870.5,16630,4105.0,54750
1188,2026-09-11,3274.0,14238.5,1854.0,16270,4015.0,53755
1189,2026-09-14,3311.0,14044.0,1834.0,16205,3951.0,52200
1190,2026-09-15,3260.0,14045.0,1829.0,16135,3922.0,51800
1191,2026-09-16,3310.0,14227.0,1848.0,16090,3937.0,52310


westmetall_db


,date,aluminium,copper,lead,nickel,zink,tin
1187,2026-09-10,3343.0,14390.0,1870.5,16630,4105.0,54750
1188,2026-09-11,3274.0,14238.5,1854.0,16270,4015.0,53755
1189,2026-09-14,3311.0,14044.0,1834.0,16205,3951.0,52200
1190,2026-09-15,3260.0,14045.0,1829.0,16135,3922.0,51800
1191,2026-09-16,3310.0,14227.0,1848.0,16090,3937.0,52310


kitco_db


...

,date,gold,silver,platinum,palladium
910,2026-09-11,11972.94,179.68,5130.31,3689.59
911,2026-09-12,11825.66,178.00,4905.31,3523.49
912,2026-09-15,11893.20,173.10,4898.41,3581.18
913,2026-09-16,11556.39,170.11,4777.36,3494.87
914,2026-09-17,11626.37,170.95,4811.40,3528.92


nbk_tenge_db


,date,Числовое значение,ДОЛЛАР США
1684,2026-09-13,1,450.91
1685,2026-09-14,1,450.91
1686,2026-09-15,1,447.80
1687,2026-09-16,1,447.88
1688,2026-09-17,1,444.88


shmet_historical_db


,date,price,unit
0,2020-01-10,48605,Yuan/MT
1,2020-01-14,48990,Yuan/MT
2,2020-01-15,49060,Yuan/MT
3,2020-01-16,48950,Yuan/MT
4,2020-01-17,48930,Yuan/MT
